# 머신 러닝 교과서 3판

# 8장 - 감성 분석에 머신 러닝 적용

**아래 링크를 통해 이 노트북을 주피터 노트북 뷰어(nbviewer.jupyter.org)로 보거나 구글 코랩(colab.research.google.com)에서 실행할 수 있습니다.**

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://nbviewer.org/github/rickiepark/python-machine-learning-book-3rd-edition/blob/master/ch08/ch08.ipynb"><img src="https://jupyter.org/assets/share.png" width="60" />주피터 노트북 뷰어로 보기</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/rickiepark/python-machine-learning-book-3rd-edition/blob/master/ch08/ch08.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩(Colab)에서 실행하기</a>
  </td>
</table>

### 목차

- 텍스트 처리용 IMDb 영화 리뷰 데이터 준비
    - 영화 리뷰 데이터셋 구하기
    -영화 리뷰 데이터셋을 더 간편한 형태로 전처리
- BoW 모델 소개
    - 단어를 특성 벡터로 변환
    - tf-idf를 사용하여 단어 적합성 평가
    - 텍스트 데이터 정제
    - 문서를 토큰으로 나누기
- 문서 분류를 위한 로지스틱 회귀 모델 훈련
- 대용량 데이터 처리: 온라인 알고리즘과 외부 메모리 학습
- 잠재 디리클레 할당을 사용한 토픽 모델링
    - LDA를 사용한 텍스트 문서 분해
    - 사이킷런의 LDA
- 요약

# 8.1 텍스트 처리용 IMDb 영화 리뷰 데이터 준비

## 8.1.1 영화 리뷰 데이터셋 구하기

IMDB 영화 리뷰 데이터셋은 [http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz](http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz)에서 내려받을 수 있습니다. 다운로드된 후 파일 압축을 해제합니다.

A) 리눅스(Linux)나 macOS를 사용하면 새로운 터미널(Terminal) 윈도우를 열고 `cd` 명령으로 다운로드 디렉터리로 이동하여 다음 명령을 실행하세요.

`tar -zxf aclImdb_v1.tar.gz`

B) 윈도(Windows)를 사용하면 7-Zip(http://www.7-zip.org) 같은 무료 압축 유틸리티를 설치하여 다운로드한 파일의 압축을 풀 수 있습니다.

**다음처럼 파이썬에서 다운로드하고 압축을 풀 수도 있습니다:**

In [1]:
import os
import sys
import tarfile
import time
import urllib.request


source = 'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
target = 'aclImdb_v1.tar.gz'


def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = time.time()
        return
    duration = time.time() - start_time
    progress_size = int(count * block_size)
    speed = progress_size / (1024.**2 * duration)
    percent = count * block_size * 100. / total_size

    sys.stdout.write("\r%d%% | %d MB | %.2f MB/s | %d sec elapsed" %
                    (percent, progress_size / (1024.**2), speed, duration))
    sys.stdout.flush()


if not os.path.isdir('aclImdb') and not os.path.isfile('aclImdb_v1.tar.gz'):
    urllib.request.urlretrieve(source, target, reporthook)

In [2]:
if not os.path.isdir('aclImdb'):

    with tarfile.open(target, 'r:gz') as tar:
        tar.extractall()

## 8.1.2 영화 리뷰 데이터셋을 더 간편한 형태로 전처리

`pyprind`는 주피터 노트북에서 진행바를 출력하기 위한 유틸리티입니다. `pyprind` 패키지를 설치하려면 다음 셀을 실행하세요.

In [3]:
%pip install pyprind

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pyprind
import pandas as pd
import os

# `basepath`를 압축 해제된 영화 리뷰 데이터셋이 있는
# 디렉토리로 바꾸세요

basepath = 'aclImdb'

labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000)
# df = pd.DataFrame()
rows = []
for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file),
                      'r', encoding='utf-8') as infile:
                txt = infile.read()
            rows.append([txt, labels[l]])
            # df = df.append([[txt, labels[l]]],
            #                ignore_index=True)
            pbar.update()
df = pd.DataFrame(rows, columns=['review', 'sentiment'])
# df.columns = ['review', 'sentiment']

데이터프레임을 섞습니다:

In [5]:
import numpy as np

np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))